In [1]:
%pip install -U langchain-core langchain-community langchain-classic sqlalchemy pymysql openai python-dotenv sqlglot pandas

  Using cached langchain_core-1.6.10-py3-none-any.whl.metadata (4.8 kB)
Using cached langchain_core-1.6.10-py3-none-any.whl (573 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.9
    Uninstalling langchain-core-1.6.9:
      Successfully uninstalled langchain-core-1.6.9
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 1.2.2 requires huggingface-hub<2.0.0,>=0.33.4, but you have huggingface-hub 2.2.0 which is incompatible.
langchain-openai 1.1.9 requires openai<3.0.0,>=1.109.1, but you have openai 3.29.0 which is incompatible.


In [2]:
import os
import re
import json

import pandas as pd
import sqlglot
from sqlglot import exp

from dotenv import load_dotenv
from openai import OpenAI

from langchain_community.utilities import SQLDatabase
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from sqlalchemy import URL

C:\Users\chint\AppData\Local\Temp\ipykernel_3252\468591616.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [3]:
from pathlib import Path
import os

env_path = Path.cwd() / ".env"

placeholder = '''OPENROUTER_API_KEY=your_openrouter_api_key_here
GEMINI_API_KEY=your_gemini_api_key_here
MYSQL_USER=root
MYSQL_PASSWORD=your_mysql_password_here
MYSQL_HOST=localhost
MYSQL_PORT=3306
MYSQL_DATABASE=text_to_sql
'''

if not env_path.exists():
    env_path.write_text(placeholder, encoding="utf-8")
    print("Created template .env file")
else:
    print(".env file already exists")


In [4]:
from pathlib import Path

print("Current directory:", Path.cwd())
print("Does .env exist?", (Path.cwd() / ".env").exists())
print("Files:", [p.name for p in Path.cwd().iterdir()])

Current directory: C:\Users\chint\SE_project
Does .env exist? True
Files: ['.env', '.ipynb_checkpoints', 'text_to_sql_conversion.ipynb']


In [29]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
model_name = os.getenv("OPENROUTER_MODEL", "openrouter/free")

if not api_key:
    raise ValueError("OPENROUTER_API_KEY is missing from .env")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

def call_llm(prompt):
    if hasattr(prompt, "to_string"):
        prompt = prompt.to_string()
    else:
        prompt = str(prompt)

    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=1024,
        temperature=0
    )

    choice = response.choices[0]
    content = choice.message.content

    print("Finish reason:", choice.finish_reason)

    if content is None or not content.strip():
        raise ValueError(
            f"LLM returned empty content. "
            f"Finish reason: {choice.finish_reason}. "
            f"Try another model or increase the token limit."
        )

    return content.strip()

In [30]:
required_vars = [
    "MYSQL_USER",
    "MYSQL_PASSWORD",
    "MYSQL_HOST",
    "MYSQL_PORT",
    "MYSQL_DATABASE"
]

missing_vars = [
    name for name in required_vars
    if not os.getenv(name)
]

if missing_vars:
    raise ValueError(
        f"Missing environment variables: {', '.join(missing_vars)}"
    )

mysql_url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("MYSQL_USER"),
    password=os.getenv("MYSQL_PASSWORD"),
    host=os.getenv("MYSQL_HOST"),
    port=int(os.getenv("MYSQL_PORT")),
    database=os.getenv("MYSQL_DATABASE")
)

db = SQLDatabase.from_uri(
    mysql_url,
    sample_rows_in_table_info=2
)

print("Connected:", db.dialect)
print("Tables:", db.get_usable_table_names())

Connected: mysql
Tables: ['2017_budgets', 'customers', 'products', 'regions', 'sales_order', 'sales_orders_old', 'state_regions']


In [31]:
def get_schema():
    return db.get_table_info()

schema = get_schema()

print(schema)


CREATE TABLE `2017_budgets` (
	`Product Name` TEXT, 
	`2017 Budgets` DOUBLE
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
2 rows from 2017_budgets table:
Product Name	2017 Budgets
Product 1	3016489.2089999998
Product 2	3050087.5649999999
*/


CREATE TABLE customers (
	`Customer Index` INTEGER, 
	`Customer Names` TEXT
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
2 rows from customers table:
Customer Index	Customer Names
1	Geiss Company
2	Jaxbean Group
*/


CREATE TABLE products (
	`Index` INTEGER, 
	`Product Name` TEXT
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
2 rows from products table:
Index	Product Name
1	Product 1
2	Product 2
*/


CREATE TABLE regions (
	id INTEGER, 
	name TEXT, 
	county TEXT, 
	state_code TEXT, 
	state TEXT, 
	type TEXT, 
	latitude DOUBLE, 
	longitude DOUBLE, 
	area_code INTEGER, 
	population INTEGER, 
	households INTEGER, 
	median_income INTEGER, 
	land_area INTEGER, 
	water_area INTEGER

In [32]:
sql_prompt = ChatPromptTemplate.from_template("""
You are a MySQL Text-to-SQL assistant.

Convert the user's question into exactly one MySQL SELECT query.

Rules:
1. Use only tables and columns present in the schema.
2. Use correct MySQL syntax.
3. Use backticks around identifiers containing spaces or reserved words.
4. Do not invent columns, tables, or relationships.
5. Do not generate INSERT, UPDATE, DELETE, DROP, ALTER, CREATE,
   TRUNCATE, or other data-modification statements.
6. Do not generate multiple statements.
7. Return only the SQL query, without Markdown or explanations.
8. If the question cannot be answered from the schema, return:
   SELECT 1 WHERE FALSE;

Database schema:
{schema}

User question:
{question}

SQL query:
""")

In [33]:
llm = RunnableLambda(call_llm)

sql_chain = (
    RunnablePassthrough.assign(
        schema=lambda _: get_schema()
    )
    | sql_prompt
    | llm
    | StrOutputParser()
)

print("Text-to-SQL chain created successfully")


Text-to-SQL chain created successfully


In [34]:
def generate_sql(question):
    response = sql_chain.invoke({
        "question": question
    })

    return response.strip()

In [35]:
question = "What is the total line total for Geiss Company?"

generated_sql = generate_sql(question)

print("Question:", question)
print("Generated SQL:", generated_sql)

Finish reason: stop
Question: What is the total line total for Geiss Company?
Generated SQL: SELECT SUM(`sales_order`.`Line Total`) FROM `sales_order` JOIN `customers` ON `sales_order`.`Customer Name Index` = `customers`.`Customer Index` WHERE `customers`.`Customer Names` = 'Geiss Company';


In [36]:
print("Testing OpenRouter...")

test_response = call_llm(
    "Return only this exact text: SELECT 1;"
)

print("Final test response:", test_response)

Testing OpenRouter...
Finish reason: stop
Final test response: SELECT 1;


In [37]:
def extract_sql(response):
    if not response or not response.strip():
        raise ValueError("The LLM returned an empty response")

    response = response.strip()

    match = re.fullmatch(
        r"```(?:sql)?\s*(.*?)\s*```",
        response,
        flags=re.IGNORECASE | re.DOTALL
    )

    if match:
        response = match.group(1).strip()

    if response.lower().startswith("sql query:"):
        response = response[len("sql query:"):].strip()

    return response

In [38]:
def validate_sql(query):
    if not query or not query.strip():
        return False, "SQL query is empty"

    try:
        statements = sqlglot.parse(query, read="mysql")

        if len(statements) != 1 or statements[0] is None:
            return False, "Only one SQL statement is allowed"

        statement = statements[0]

        if not isinstance(statement, exp.Select):
            return False, "Only SELECT queries are allowed"

        if statement.args.get("into") is not None:
            return False, "SELECT INTO is not allowed"

        forbidden_functions = {
            "SLEEP",
            "BENCHMARK",
            "LOAD_FILE",
            "INTO_OUTFILE",
            "INTO_DUMPFILE"
        }

        for node in statement.walk():
            if isinstance(node, exp.Func):
                function_name = node.sql_name().upper()

                if function_name in forbidden_functions:
                    return False, f"Disallowed function: {function_name}"

        return True, "SQL validation successful"

    except Exception as e:
        return False, f"Invalid SQL syntax: {e}"

In [39]:
test_queries = [
    "SELECT COUNT(*) FROM customers",
    "DELETE FROM customers",
    "UPDATE customers SET `Customer Names` = 'Test'",
    "SELECT * FROM customers; DROP TABLE customers"
]

for query in test_queries:
    valid, message = validate_sql(query)
    print(query)
    print("Valid:", valid)
    print("Message:", message)
    print("-" * 50)

SELECT COUNT(*) FROM customers
Valid: True
Message: SQL validation successful
--------------------------------------------------
DELETE FROM customers
Valid: False
Message: Only SELECT queries are allowed
--------------------------------------------------
UPDATE customers SET `Customer Names` = 'Test'
Valid: False
Message: Only SELECT queries are allowed
--------------------------------------------------
SELECT * FROM customers; DROP TABLE customers
Valid: False
Message: Only one SQL statement is allowed
--------------------------------------------------


In [40]:
def execute_sql(query):
    is_valid, message = validate_sql(query)

    if not is_valid:
        return {
            "success": False,
            "sql": query,
            "result": None,
            "message": message
        }

    try:
        result = db.run(query)

        return {
            "success": True,
            "sql": query,
            "result": result,
            "message": "Query executed successfully"
        }

    except Exception as e:
        return {
            "success": False,
            "sql": query,
            "result": None,
            "message": f"Database error: {e}"
        }

In [41]:
def generate_answer(question, sql, result):
    prompt = f"""
You are a data analyst assistant.

Answer the user's question using only the database result below.
Do not invent numbers or claim information that is not present.
If the result is empty, clearly state that no matching records were found.
Keep the answer concise and easy to understand.

User question:
{question}

SQL executed:
{sql}

Database result:
{result}

Answer:
"""

    return call_llm(prompt)

In [42]:
def answer_database_question(question):
    try:
        raw_sql = generate_sql(question)
        query = extract_sql(raw_sql)

        execution = execute_sql(query)

        if not execution["success"]:
            return {
                "question": question,
                **execution,
                "answer": execution["message"]
            }

        answer = generate_answer(
            question,
            query,
            execution["result"]
        )

        return {
            "question": question,
            "success": True,
            "sql": query,
            "result": execution["result"],
            "answer": answer,
            "message": "Query completed successfully"
        }

    except Exception as e:
        return {
            "question": question,
            "success": False,
            "sql": None,
            "result": None,
            "answer": "Unable to process the question.",
            "message": str(e)
        }

In [43]:
question = "How many customers are there?"


print(sql_chain.invoke({"question": question}))

Finish reason: stop
SELECT COUNT(*) FROM customers;


In [44]:
response = answer_database_question(
    "What is the total line total for Geiss Company?"
)

print("Question:", response["question"])
print("Success:", response["success"])
print("Generated SQL:", response["sql"])
print("Database Result:", response["result"])
print("Answer:", response["answer"])
print("Status:", response["message"])

Finish reason: stop
Finish reason: stop
Question: What is the total line total for Geiss Company?
Success: True
Generated SQL: SELECT SUM(`Line Total`) AS total_line_total FROM sales_order JOIN customers ON sales_order.`Customer Name Index` = customers.`Customer Index` WHERE customers.`Customer Names` = 'Geiss Company';
Database Result: [(2527776.0000000005,)]
Answer: User Safety: safe
Status: Query completed successfully


In [45]:
questions = [
    "How many customers are there?",
    "What are the top 5 products by total sales?",
    "What is the total sales amount for each channel?",
    "Which customer has the highest total sales?"
]

for question in questions:
    response = answer_database_question(question)

    print("\nQuestion:", question)
    print("Success:", response["success"])
    print("SQL:", response["sql"])
    print("Answer:", response["answer"])
    print("-" * 60)

Finish reason: stop
Finish reason: stop

Question: How many customers are there?
Success: True
SQL: SELECT COUNT(*) FROM `customers`;
Answer: There are 133 customers.
------------------------------------------------------------
Finish reason: length

Question: What are the top 5 products by total sales?
Success: False
SQL: None
Answer: Unable to process the question.
------------------------------------------------------------
Finish reason: stop
Finish reason: stop

Question: What is the total sales amount for each channel?
Success: True
SQL: SELECT `Channel`, SUM(`Line Total`) AS total_sales_amount FROM `sales_order` GROUP BY `Channel`;
Answer: Wholesale: 280171126.19999933  
Distributor: 162194022.0999997  
Export: 75540463.20000024
------------------------------------------------------------
Finish reason: stop
Finish reason: stop

Question: Which customer has the highest total sales?
Success: True
SQL: SELECT c.`Customer Names` AS CustomerName, SUM(s.`Line Total`) AS TotalSales
FR

In [46]:

import pandas as pd
import time
from datetime import datetime

test_cases = [
    # EASY: basic queries
    {
        "id": 1,
        "difficulty": "Easy",
        "question": "How many customers are there?"
    },
    {
        "id": 2,
        "difficulty": "Easy",
        "question": "How many orders are there in sales_order?"
    },
    {
        "id": 3,
        "difficulty": "Easy",
        "question": "What is the total sales amount?"
    },
    {
        "id": 4,
        "difficulty": "Easy",
        "question": "What is the average order quantity?"
    },
    {
        "id": 5,
        "difficulty": "Easy",
        "question": "What is the total sales amount for each channel?"
    },

    # MEDIUM: filtering, grouping and sorting
    {
        "id": 6,
        "difficulty": "Medium",
        "question": "What are the top 5 products by total sales?"
    },
    {
        "id": 7,
        "difficulty": "Medium",
        "question": "Which customer has the highest total sales?"
    },
    {
        "id": 8,
        "difficulty": "Medium",
        "question": "Show the 10 customers with the highest total sales."
    },
    {
        "id": 9,
        "difficulty": "Medium",
        "question": "Which sales channel has the highest total revenue?"
    },
    {
        "id": 10,
        "difficulty": "Medium",
        "question": "Show the number of orders for each sales channel."
    },
    {
        "id": 11,
        "difficulty": "Medium",
        "question": "Which products have a total sales amount greater than 1000000?"
    },
    {
        "id": 12,
        "difficulty": "Medium",
        "question": "What is the total profit for each channel, calculated as Line Total minus Total Unit Cost times Order Quantity?"
    },
    {
        "id": 13,
        "difficulty": "Medium",
        "question": "Find the customers whose total sales exceed 100000."
    },
    {
        "id": 14,
        "difficulty": "Medium",
        "question": "What is the highest and lowest unit price in sales_order?"
    },

    # HARD: joins, ratios, ranking and nested logic
    {
        "id": 15,
        "difficulty": "Hard",
        "question": "For each customer, calculate total sales and total order quantity, and return the top 5 customers by total sales."
    },
    {
        "id": 16,
        "difficulty": "Hard",
        "question": "What percentage of total sales comes from each sales channel?"
    },
    {
        "id": 17,
        "difficulty": "Hard",
        "question": "Find products whose total sales are above the average total sales per product."
    },
    {
        "id": 18,
        "difficulty": "Hard",
        "question": "Which customers have placed more orders than the average number of orders per customer?"
    },
    {
        "id": 19,
        "difficulty": "Hard",
        "question": "Rank sales channels from highest to lowest total sales and show each channel's percentage contribution to overall sales."
    },
    {
        "id": 20,
        "difficulty": "Hard",
        "question": "Compare the total sales in sales_order with sales_orders_old and calculate the difference."
    },

    # CHALLENGE: ambiguous or schema-dependent questions
    {
        "id": 21,
        "difficulty": "Challenge",
        "question": "For each product, identify the customer who purchased the largest quantity of it."
    },
    {
        "id": 22,
        "difficulty": "Challenge",
        "question": "Find customers whose total sales are greater than the average total sales across all customers."
    },
    {
        "id": 23,
        "difficulty": "Challenge",
        "question": "Calculate monthly sales growth percentage compared with the previous month."
    },
    {
        "id": 24,
        "difficulty": "Challenge",
        "question": "Find the top 3 products by revenue within each sales channel."
    },
    {
        "id": 25,
        "difficulty": "Challenge",
        "question": "Are there any customers with no orders? Return their customer names."
    }
]

results = []

for test in test_cases:
    start_time = time.time()

    row = {
        "Test ID": test["id"],
        "Difficulty": test["difficulty"],
        "Question": test["question"],
        "Success": False,
        "SQL": None,
        "Answer": None,
        "Error": None,
        "Time (seconds)": None
    }

    try:
        output = answer_database_question(test["question"])

        row["Success"] = bool(output.get("success", False))
        row["SQL"] = output.get("sql")
        row["Answer"] = output.get("answer")

        if not row["Success"]:
            row["Error"] = (
                output.get("error")
                or output.get("message")
                or "Generation or execution failed"
            )

    except Exception as e:
        row["Error"] = f"{type(e).__name__}: {e}"

    row["Time (seconds)"] = round(time.time() - start_time, 2)
    results.append(row)

    status = "PASS" if row["Success"] else "FAIL"
    print(f"\n[{status}] Test {test['id']} | {test['difficulty']}")
    print("Question:", test["question"])
    print("SQL:", row["SQL"])
    print("Answer:", row["Answer"])
    if row["Error"]:
        print("Error:", row["Error"])
    print("Time:", row["Time (seconds)"], "seconds")

    time.sleep(0.2)

evaluation_df = pd.DataFrame(results)

print("\n========== EVALUATION SUMMARY ==========")
print("Total tests:", len(evaluation_df))
print("Successful executions:", int(evaluation_df["Success"].sum()))
print("Failed executions:", int((~evaluation_df["Success"]).sum()))
print(
    "Execution success rate:",
    round(evaluation_df["Success"].mean() * 100, 2),
    "%"
)
print("\nSuccess rate by difficulty:")
print(
    evaluation_df.groupby("Difficulty")["Success"]
    .agg(["sum", "count", "mean"])
    .assign(success_rate=lambda x: (x["mean"] * 100).round(2))
    [["sum", "count", "success_rate"]]
)

evaluation_df.to_csv("text_to_sql_evaluation.csv", index=False)
print("\nSaved results to text_to_sql_evaluation.csv")


Finish reason: stop
Finish reason: stop

[PASS] Test 1 | Easy
Question: How many customers are there?
SQL: SELECT COUNT(*) FROM `customers`;
Answer: User Safety: safe
Time: 12.31 seconds
Finish reason: stop
Finish reason: stop

[PASS] Test 2 | Easy
Question: How many orders are there in sales_order?
SQL: SELECT COUNT(*) FROM `sales_order`;
Answer: There are 26,909 orders in the sales_order table.
Time: 5.56 seconds
Finish reason: length

[FAIL] Test 3 | Easy
Question: What is the total sales amount?
SQL: None
Answer: Unable to process the question.
Error: LLM returned empty content. Finish reason: length. Try another model or increase the token limit.
Time: 12.97 seconds
Finish reason: stop
Finish reason: stop

[PASS] Test 4 | Easy
Question: What is the average order quantity?
SQL: SELECT AVG(`Order Quantity`) FROM sales_order;
Answer: The average order quantity is **8.4497**.
Time: 7.57 seconds
Finish reason: stop
Finish reason: stop

[PASS] Test 5 | Easy
Question: What is the total s

In [47]:

failed_tests = evaluation_df[evaluation_df["Success"] == False]

display(
    failed_tests[
        ["Test ID", "Difficulty", "Question", "SQL", "Error"]
    ]
)


,Test ID,Difficulty,Question,SQL,Error
2,3,Easy,What is the total sales amount?,NaN,LLM returned empty content. Finish reason: len...
5,6,Medium,What are the top 5 products by total sales?,NaN,LLM returned empty content. Finish reason: len...
6,7,Medium,Which customer has the highest total sales?,NaN,LLM returned empty content. Finish reason: len...
7,8,Medium,Show the 10 customers with the highest total s...,NaN,LLM returned empty content. Finish reason: len...
8,9,Medium,Which sales channel has the highest total reve...,NaN,LLM returned empty content. Finish reason: len...
10,11,Medium,Which products have a total sales amount great...,NaN,LLM returned empty content. Finish reason: len...
12,13,Medium,Find the customers whose total sales exceed 10...,NaN,LLM returned empty content. Finish reason: len...
14,15,Hard,"For each customer, calculate total sales and t...",NaN,LLM returned empty content. Finish reason: len...
15,16,Hard,What percentage of total sales comes from each...,"SELECT \n cs.`Channel`,\n SUM(cs.`Line T...",Invalid SQL syntax: Expected table name but go...
16,17,Hard,Find products whose total sales are above the ...,NaN,LLM returned empty content. Finish reason: len...


In [48]:

for _, row in failed_tests.iterrows():
    print("=" * 80)
    print(f"Test {row['Test ID']} | {row['Difficulty']}")
    print("Question:", row["Question"])
    print("Generated SQL:", row["SQL"])
    print("Error:", row["Error"])
    print("Answer:", row["Answer"])


Test 3 | Easy
Question: What is the total sales amount?
Generated SQL: nan
Error: LLM returned empty content. Finish reason: length. Try another model or increase the token limit.
Answer: Unable to process the question.
Test 6 | Medium
Question: What are the top 5 products by total sales?
Generated SQL: nan
Error: LLM returned empty content. Finish reason: length. Try another model or increase the token limit.
Answer: Unable to process the question.
Test 7 | Medium
Question: Which customer has the highest total sales?
Generated SQL: nan
Error: LLM returned empty content. Finish reason: length. Try another model or increase the token limit.
Answer: Unable to process the question.
Test 8 | Medium
Question: Show the 10 customers with the highest total sales.
Generated SQL: nan
Error: LLM returned empty content. Finish reason: length. Try another model or increase the token limit.
Answer: Unable to process the question.
Test 9 | Medium
Question: Which sales channel has the highest total re